# 02 — Longitudinal Patient Journey

## EHR Patient Journey & Clinical Outcomes Analytics

### Objective

Reconstruct longitudinal patient journeys from EHR-style data by linking encounters with diagnoses, observations, medications, and procedures.

The analysis will examine how patients move through the healthcare system over time and create an encounter-level analytical dataset that can support downstream clinical and operational analyses.

### Patient Journey Framework

**Patient → Encounter → Diagnosis → Clinical Observations → Medication → Procedure → Subsequent Care**

This framework reflects the relational nature of electronic health record data and allows clinical events to be interpreted within the context of individual episodes of care.

In [14]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

DATA_PATH = Path("../data/raw")

patients = pd.read_csv(DATA_PATH / "patients.csv")
encounters = pd.read_csv(DATA_PATH / "encounters.csv")
conditions = pd.read_csv(DATA_PATH / "conditions.csv")
observations = pd.read_csv(DATA_PATH / "observations.csv")
medications = pd.read_csv(DATA_PATH / "medications.csv")
procedures = pd.read_csv(DATA_PATH / "procedures.csv")

print("EHR tables loaded successfully.")
print(f"Patients:     {len(patients):,}")
print(f"Encounters:   {len(encounters):,}")
print(f"Conditions:   {len(conditions):,}")
print(f"Observations: {len(observations):,}")
print(f"Medications:  {len(medications):,}")
print(f"Procedures:   {len(procedures):,}")

EHR tables loaded successfully.
Patients:     108
Encounters:   5,571
Conditions:   3,517
Observations: 68,648
Medications:  3,850
Procedures:   15,884


## 2. Build the Encounter Timeline

Encounters form the backbone of the longitudinal patient journey. Each encounter represents an interaction between a patient and the healthcare system.

To reconstruct the sequence of care, encounter timestamps are converted to datetime format, duration is calculated, and encounters are ordered chronologically within each patient.

In [15]:
# Convert encounter timestamps to datetime
encounters["START"] = pd.to_datetime(encounters["START"], errors="coerce")
encounters["STOP"] = pd.to_datetime(encounters["STOP"], errors="coerce")

# Calculate encounter duration
encounters["duration_hours"] = (
    encounters["STOP"] - encounters["START"]
).dt.total_seconds() / 3600

# Sort chronologically within each patient
encounter_timeline = (
    encounters
    .sort_values(["PATIENT", "START"])
    .reset_index(drop=True)
)

print(f"Total encounters: {len(encounter_timeline):,}")
print(f"Unique patients: {encounter_timeline['PATIENT'].nunique():,}")
print(
    "Encounter date range:",
    encounter_timeline["START"].min(),
    "to",
    encounter_timeline["START"].max()
)

encounter_timeline[
    [
        "Id",
        "PATIENT",
        "START",
        "STOP",
        "ENCOUNTERCLASS",
        "DESCRIPTION",
        "duration_hours"
    ]
].head(15)

Total encounters: 5,571
Unique patients: 108
Encounter date range: 1942-11-26 20:44:21+00:00 to 2026-08-16 23:50:14+00:00


,Id,PATIENT,START,STOP,ENCOUNTERCLASS,DESCRIPTION,duration_hours
0,00bcbfc6-c7b9-bd0e-0c7a-bfa7c60525a3,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,1999-03-16 01:16:49+00:00,1999-03-16 01:50:20+00:00,wellness,General examination of patient (procedure),0.558611
1,00bcbfc6-c7b9-bd0e-3c9c-04c7c31d4e70,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2006-03-28 01:16:49+00:00,2006-03-28 01:50:31+00:00,wellness,General examination of patient (procedure),0.561667
2,00bcbfc6-c7b9-bd0e-bd6c-67a2c4630b2a,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2012-04-03 01:16:49+00:00,2012-04-03 02:06:33+00:00,wellness,General examination of patient (procedure),0.828889
3,00bcbfc6-c7b9-bd0e-a616-fd9cb0db9328,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2013-04-09 01:16:49+00:00,2013-04-09 01:57:45+00:00,wellness,General examination of patient (procedure),0.682222
4,00bcbfc6-c7b9-bd0e-b3e0-fded30c067fb,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2016-04-26 01:16:49+00:00,2016-04-26 02:10:34+00:00,wellness,General examination of patient (procedure),0.895833
5,00bcbfc6-c7b9-bd0e-5508-fed429d48fda,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-04-25 01:16:49+00:00,2017-04-25 01:56:04+00:00,outpatient,Patient encounter procedure (procedure),0.654167
6,00bcbfc6-c7b9-bd0e-9492-f3f5df2feff3,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-05-09 01:16:49+00:00,2017-05-09 01:48:00+00:00,wellness,General examination of patient (procedure),0.519722
7,00bcbfc6-c7b9-bd0e-d4a2-6d081b2d5302,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-06-13 01:16:49+00:00,2017-06-13 01:31:49+00:00,ambulatory,Prenatal initial visit (regime/therapy),0.250000
8,00bcbfc6-c7b9-bd0e-fbed-dbb2ed31ce69,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-07-07 08:16:49+00:00,2017-07-07 08:31:49+00:00,ambulatory,Encounter for symptom (procedure),0.250000
9,00bcbfc6-c7b9-bd0e-d509-3138062c5530,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-07-11 01:16:49+00:00,2017-07-11 01:31:49+00:00,ambulatory,Prenatal visit (regime/therapy),0.250000


## 3. Previous and Next Encounter

A longitudinal EHR analysis requires understanding each encounter in the context of the patient's previous and subsequent healthcare utilization.

For each encounter, the previous and next encounter are identified, along with the time elapsed between episodes of care. These features allow the patient journey to be analyzed as a sequence rather than as isolated healthcare events.

In [16]:
# Previous encounter information
encounter_timeline["previous_encounter_class"] = (
    encounter_timeline
    .groupby("PATIENT")["ENCOUNTERCLASS"]
    .shift(1)
)

encounter_timeline["previous_encounter_date"] = (
    encounter_timeline
    .groupby("PATIENT")["START"]
    .shift(1)
)

# Next encounter information
encounter_timeline["next_encounter_class"] = (
    encounter_timeline
    .groupby("PATIENT")["ENCOUNTERCLASS"]
    .shift(-1)
)

encounter_timeline["next_encounter_date"] = (
    encounter_timeline
    .groupby("PATIENT")["START"]
    .shift(-1)
)

# Time between encounters
encounter_timeline["days_since_previous"] = (
    encounter_timeline["START"]
    - encounter_timeline["previous_encounter_date"]
).dt.total_seconds() / 86400

encounter_timeline["days_to_next"] = (
    encounter_timeline["next_encounter_date"]
    - encounter_timeline["START"]
).dt.total_seconds() / 86400

journey_cols = [
    "PATIENT",
    "START",
    "ENCOUNTERCLASS",
    "DESCRIPTION",
    "previous_encounter_class",
    "days_since_previous",
    "next_encounter_class",
    "days_to_next"
]

encounter_timeline[journey_cols].head(20)

,PATIENT,START,ENCOUNTERCLASS,DESCRIPTION,previous_encounter_class,days_since_previous,next_encounter_class,days_to_next
0,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,1999-03-16 01:16:49+00:00,wellness,General examination of patient (procedure),NaN,NaN,wellness,2569.000000
1,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2006-03-28 01:16:49+00:00,wellness,General examination of patient (procedure),wellness,2569.000000,wellness,2198.000000
2,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2012-04-03 01:16:49+00:00,wellness,General examination of patient (procedure),wellness,2198.000000,wellness,371.000000
3,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2013-04-09 01:16:49+00:00,wellness,General examination of patient (procedure),wellness,371.000000,wellness,1113.000000
4,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2016-04-26 01:16:49+00:00,wellness,General examination of patient (procedure),wellness,1113.000000,outpatient,364.000000
5,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-04-25 01:16:49+00:00,outpatient,Patient encounter procedure (procedure),wellness,364.000000,wellness,14.000000
6,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-05-09 01:16:49+00:00,wellness,General examination of patient (procedure),outpatient,14.000000,ambulatory,35.000000
7,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-06-13 01:16:49+00:00,ambulatory,Prenatal initial visit (regime/therapy),wellness,35.000000,ambulatory,24.291667
8,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-07-07 08:16:49+00:00,ambulatory,Encounter for symptom (procedure),ambulatory,24.291667,ambulatory,3.708333
9,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2017-07-11 01:16:49+00:00,ambulatory,Prenatal visit (regime/therapy),ambulatory,3.708333,ambulatory,28.000000


## 4. Care Setting Transitions

Healthcare utilization can be represented as transitions between care settings. Examining these transitions helps identify common pathways through the health system and episodes involving escalation to acute care.

This analysis summarizes how patients move from one encounter class to the next across their longitudinal record.

In [17]:
# Exclude the final encounter for each patient,
# since it has no subsequent encounter
transitions = (
    encounter_timeline
    .dropna(subset=["next_encounter_class"])
    .groupby(
        ["ENCOUNTERCLASS", "next_encounter_class"]
    )
    .size()
    .reset_index(name="transition_count")
)

# Calculate percentage within each originating care setting
transitions["transition_pct"] = (
    transitions["transition_count"]
    / transitions.groupby("ENCOUNTERCLASS")["transition_count"].transform("sum")
    * 100
)

transitions = transitions.sort_values(
    "transition_count",
    ascending=False
).reset_index(drop=True)

transitions.head(20).round({"transition_pct": 1})

,ENCOUNTERCLASS,next_encounter_class,transition_count,transition_pct
0,ambulatory,ambulatory,1832,63.7
1,wellness,ambulatory,643,52.1
2,ambulatory,wellness,519,18.0
3,wellness,wellness,382,30.9
4,ambulatory,outpatient,299,10.4
5,outpatient,ambulatory,268,35.8
6,outpatient,outpatient,236,31.6
7,outpatient,wellness,198,26.5
8,wellness,outpatient,136,11.0
9,ambulatory,emergency,117,4.1


## 5. Acute Care Follow-up

Emergency encounters represent important points of acute healthcare utilization. To distinguish routine longitudinal care from potentially meaningful repeat utilization, this section examines what happens after an emergency encounter and how quickly patients return to the healthcare system.

Particular attention is given to repeat emergency encounters and short-term follow-up within 7 and 30 days.

In [18]:
# Isolate emergency encounters
emergency_journeys = encounter_timeline[
    encounter_timeline["ENCOUNTERCLASS"] == "emergency"
].copy()

print(f"Total emergency encounters: {len(emergency_journeys):,}")
print(f"Patients with an emergency encounter: {emergency_journeys['PATIENT'].nunique():,}")

# What happens after an emergency encounter?
emergency_followup = (
    emergency_journeys["next_encounter_class"]
    .value_counts(dropna=False)
    .rename_axis("next_encounter_class")
    .reset_index(name="encounters")
)

emergency_followup["percent"] = (
    emergency_followup["encounters"]
    / len(emergency_journeys)
    * 100
).round(1)

emergency_followup

Total emergency encounters: 270
Patients with an emergency encounter: 83


,next_encounter_class,encounters,percent
0,ambulatory,96,35.6
1,emergency,67,24.8
2,wellness,47,17.4
3,urgentcare,25,9.3
4,inpatient,13,4.8
5,snf,8,3.0
6,outpatient,8,3.0
7,NaN,6,2.2


In [19]:
# Repeat emergency utilization
repeat_ed = emergency_journeys[
    emergency_journeys["next_encounter_class"] == "emergency"
].copy()

repeat_ed["repeat_ed_7d"] = (
    repeat_ed["days_to_next"].between(0, 7, inclusive="both")
)

repeat_ed["repeat_ed_30d"] = (
    repeat_ed["days_to_next"].between(0, 30, inclusive="both")
)

print("REPEAT EMERGENCY UTILIZATION")
print("-" * 45)

print(f"Emergency → Emergency transitions: {len(repeat_ed):,}")

print(
    f"Repeat ED within 7 days: "
    f"{repeat_ed['repeat_ed_7d'].sum():,}"
)

print(
    f"Repeat ED within 30 days: "
    f"{repeat_ed['repeat_ed_30d'].sum():,}"
)

print(
    f"Median days to next ED encounter: "
    f"{repeat_ed['days_to_next'].median():.1f}"
)

REPEAT EMERGENCY UTILIZATION
---------------------------------------------
Emergency → Emergency transitions: 67
Repeat ED within 7 days: 63
Repeat ED within 30 days: 64
Median days to next ED encounter: 7.0


## 6. Short-Term Emergency Department Reutilization

Short-term emergency department reutilization was evaluated using all eligible emergency encounters as the denominator. A repeat ED visit was defined as a subsequent emergency encounter occurring within 7 or 30 days of the index ED encounter.

This encounter-level measure provides a more clinically interpretable estimate of acute-care reutilization than examining ED-to-ED transitions alone.

In [20]:
# Calculate encounter-level ED revisit rates

ed_index = emergency_journeys[
    emergency_journeys["next_encounter_class"].notna()
].copy()

ed_index["ed_revisit_7d"] = (
    (ed_index["next_encounter_class"] == "emergency")
    & ed_index["days_to_next"].between(0, 7, inclusive="both")
)

ed_index["ed_revisit_30d"] = (
    (ed_index["next_encounter_class"] == "emergency")
    & ed_index["days_to_next"].between(0, 30, inclusive="both")
)

n_eligible = len(ed_index)
n_7d = ed_index["ed_revisit_7d"].sum()
n_30d = ed_index["ed_revisit_30d"].sum()

rate_7d = n_7d / n_eligible * 100
rate_30d = n_30d / n_eligible * 100

print("SHORT-TERM ED REUTILIZATION")
print("-" * 45)
print(f"Eligible index ED encounters: {n_eligible:,}")
print(f"ED revisits within 7 days:  {n_7d:,} ({rate_7d:.1f}%)")
print(f"ED revisits within 30 days: {n_30d:,} ({rate_30d:.1f}%)")

SHORT-TERM ED REUTILIZATION
---------------------------------------------
Eligible index ED encounters: 264
ED revisits within 7 days:  63 (23.9%)
ED revisits within 30 days: 64 (24.2%)


## 7. Refined 30-Day ED Revisit Outcome

To accurately identify short-term emergency department reutilization, the outcome was refined to capture the next emergency encounter for each patient, regardless of whether other types of healthcare encounters occurred in between.

A 30-day ED revisit is defined as a subsequent emergency encounter occurring within 30 days of an index ED encounter.

In [21]:
# Create ED-only timeline
ed_timeline = (
    encounter_timeline[
        encounter_timeline["ENCOUNTERCLASS"] == "emergency"
    ]
    .copy()
    .sort_values(["PATIENT", "START"])
)

# Find the next ED encounter for each patient
ed_timeline["next_ed_start"] = (
    ed_timeline
    .groupby("PATIENT")["START"]
    .shift(-1)
)

# Calculate days until next ED encounter
ed_timeline["days_to_next_ed"] = (
    ed_timeline["next_ed_start"] - ed_timeline["START"]
).dt.total_seconds() / (60 * 60 * 24)

# Define revisit outcomes
ed_timeline["ed_revisit_7d"] = (
    ed_timeline["days_to_next_ed"].between(
        0, 7, inclusive="both"
    )
)

ed_timeline["ed_revisit_30d"] = (
    ed_timeline["days_to_next_ed"].between(
        0, 30, inclusive="both"
    )
)

print("REFINED ED REVISIT OUTCOME")
print("-" * 45)

print(f"Total ED encounters: {len(ed_timeline):,}")
print(
    f"ED encounters with a subsequent ED: "
    f"{ed_timeline['next_ed_start'].notna().sum():,}"
)

print(
    f"ED revisits within 7 days: "
    f"{ed_timeline['ed_revisit_7d'].sum():,} "
    f"({ed_timeline['ed_revisit_7d'].mean() * 100:.1f}%)"
)

print(
    f"ED revisits within 30 days: "
    f"{ed_timeline['ed_revisit_30d'].sum():,} "
    f"({ed_timeline['ed_revisit_30d'].mean() * 100:.1f}%)"
)

ed_timeline[
    [
        "PATIENT",
        "START",
        "DESCRIPTION",
        "next_ed_start",
        "days_to_next_ed",
        "ed_revisit_7d",
        "ed_revisit_30d"
    ]
].head(20)

REFINED ED REVISIT OUTCOME
---------------------------------------------
Total ED encounters: 270
ED encounters with a subsequent ED: 187
ED revisits within 7 days: 63 (23.3%)
ED revisits within 30 days: 80 (29.6%)


,PATIENT,START,DESCRIPTION,next_ed_start,days_to_next_ed,ed_revisit_7d,ed_revisit_30d
19,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2018-01-30 01:16:49+00:00,Obstetric emergency hospital admission (procedure),NaT,NaN,False,False
111,029c95e2-d252-d8d3-3626-7f347182d6d7,2024-06-17 16:49:02+00:00,Emergency room admission (procedure),2025-10-05 16:49:02+00:00,475.000000,False,False
116,029c95e2-d252-d8d3-3626-7f347182d6d7,2025-10-05 16:49:02+00:00,Emergency room admission (procedure),NaT,NaN,False,False
123,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-04-12 15:44:09+00:00,Emergency room admission (procedure),1995-06-25 15:44:09+00:00,74.000000,False,False
124,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-06-25 15:44:09+00:00,Emergency room admission (procedure),2000-10-16 00:58:03+00:00,1939.384653,False,False
127,031ce8e2-1443-01b4-de6b-61a4084a7166,2000-10-16 00:58:03+00:00,Emergency room admission (procedure),2009-08-25 23:12:41+00:00,3235.926829,False,False
132,031ce8e2-1443-01b4-de6b-61a4084a7166,2009-08-25 23:12:41+00:00,Emergency room admission (procedure),2013-07-07 09:55:05+00:00,1411.446111,False,False
166,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-07 09:55:05+00:00,Emergency room admission (procedure),2013-07-28 16:55:05+00:00,21.291667,False,True
167,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-28 16:55:05+00:00,Emergency room admission (procedure),2018-09-23 21:47:05+00:00,1883.202778,False,False
204,031ce8e2-1443-01b4-de6b-61a4084a7166,2018-09-23 21:47:05+00:00,Emergency room admission (procedure),NaT,NaN,False,False


## 8. Follow-up Eligibility and Right Censoring

To avoid underestimating 30-day ED reutilization, index emergency encounters must have sufficient observable follow-up time.

Encounters occurring within 30 days of the end of the available EHR observation period are excluded from the denominator for the 30-day revisit analysis.

In [22]:
# Determine the end of the observable EHR period
observation_end = encounter_timeline["START"].max()

# Require 30 complete days of potential follow-up
cutoff_30d = observation_end - pd.Timedelta(days=30)

ed_30d_eligible = ed_timeline[
    ed_timeline["START"] <= cutoff_30d
].copy()

n_total = len(ed_timeline)
n_eligible_30d = len(ed_30d_eligible)
n_censored = n_total - n_eligible_30d

n_revisit_30d = ed_30d_eligible["ed_revisit_30d"].sum()
rate_revisit_30d = n_revisit_30d / n_eligible_30d * 100

print("30-DAY ED REVISIT — FOLLOW-UP ELIGIBILITY")
print("-" * 50)

print(f"Observation period ends: {observation_end}")
print(f"30-day eligibility cutoff: {cutoff_30d}")

print(f"\nTotal ED encounters: {n_total:,}")
print(f"Eligible index ED encounters: {n_eligible_30d:,}")
print(f"Excluded due to insufficient follow-up: {n_censored:,}")

print(
    f"\n30-day ED revisits: {n_revisit_30d:,} "
    f"({rate_revisit_30d:.1f}%)"
)

30-DAY ED REVISIT — FOLLOW-UP ELIGIBILITY
--------------------------------------------------
Observation period ends: 2026-08-16 23:50:14+00:00
30-day eligibility cutoff: 2026-07-17 23:50:14+00:00

Total ED encounters: 270
Eligible index ED encounters: 263
Excluded due to insufficient follow-up: 7

30-day ED revisits: 76 (28.9%)


## 9. Build the Index ED Analytical Cohort

Each eligible emergency encounter is treated as an index encounter.

The analytical dataset combines the 30-day ED revisit outcome with patient characteristics and healthcare utilization history available at or before the index encounter.

This structure allows us to evaluate which patient and utilization characteristics are associated with short-term emergency department reutilization while avoiding data leakage.

In [23]:
# Start with eligible index ED encounters
ed_cohort = ed_30d_eligible[
    [
        "Id",
        "PATIENT",
        "START",
        "DESCRIPTION",
        "days_to_next_ed",
        "ed_revisit_7d",
        "ed_revisit_30d"
    ]
].copy()

ed_cohort = ed_cohort.rename(
    columns={
        "Id": "index_encounter_id",
        "START": "index_date",
        "DESCRIPTION": "index_description"
    }
)

print("ANALYTICAL ED COHORT")
print("-" * 45)
print(f"Rows: {len(ed_cohort):,}")
print(f"Unique patients: {ed_cohort['PATIENT'].nunique():,}")
print(f"30-day ED revisits: {ed_cohort['ed_revisit_30d'].sum():,}")
print(f"30-day revisit rate: {ed_cohort['ed_revisit_30d'].mean() * 100:.1f}%")

ed_cohort.head(10)

ANALYTICAL ED COHORT
---------------------------------------------
Rows: 263
Unique patients: 82
30-day ED revisits: 76
30-day revisit rate: 28.9%


,index_encounter_id,PATIENT,index_date,index_description,days_to_next_ed,ed_revisit_7d,ed_revisit_30d
19,00bcbfc6-c7b9-bd0e-eb8c-5a5fd719de83,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2018-01-30 01:16:49+00:00,Obstetric emergency hospital admission (procedure),NaN,False,False
111,029c95e2-d252-d8d3-22a0-9ffcfaa62a10,029c95e2-d252-d8d3-3626-7f347182d6d7,2024-06-17 16:49:02+00:00,Emergency room admission (procedure),475.000000,False,False
116,029c95e2-d252-d8d3-a200-5c4a0ef8b2df,029c95e2-d252-d8d3-3626-7f347182d6d7,2025-10-05 16:49:02+00:00,Emergency room admission (procedure),NaN,False,False
123,031ce8e2-1443-01b4-f85f-43869f746de5,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-04-12 15:44:09+00:00,Emergency room admission (procedure),74.000000,False,False
124,031ce8e2-1443-01b4-779d-a2b10a00b838,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-06-25 15:44:09+00:00,Emergency room admission (procedure),1939.384653,False,False
127,031ce8e2-1443-01b4-078b-8014a8ee8471,031ce8e2-1443-01b4-de6b-61a4084a7166,2000-10-16 00:58:03+00:00,Emergency room admission (procedure),3235.926829,False,False
132,031ce8e2-1443-01b4-1069-fa62b19913c4,031ce8e2-1443-01b4-de6b-61a4084a7166,2009-08-25 23:12:41+00:00,Emergency room admission (procedure),1411.446111,False,False
166,031ce8e2-1443-01b4-397a-909e34d396da,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-07 09:55:05+00:00,Emergency room admission (procedure),21.291667,False,True
167,031ce8e2-1443-01b4-3781-2dfe0466e1b7,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-28 16:55:05+00:00,Emergency room admission (procedure),1883.202778,False,False
204,031ce8e2-1443-01b4-40d5-34fd89da922b,031ce8e2-1443-01b4-de6b-61a4084a7166,2018-09-23 21:47:05+00:00,Emergency room admission (procedure),NaN,False,False


## 10. Patient Demographics at the Index ED Encounter

Patient age is calculated at the time of each eligible index emergency department encounter rather than using a fixed age.

This preserves the longitudinal structure of the EHR data and allows the same patient to contribute encounters at different ages.

In [24]:
# Prepare patient demographic information
patient_demo = patients[
    ["Id", "BIRTHDATE", "GENDER"]
].copy()

patient_demo["BIRTHDATE"] = pd.to_datetime(
    patient_demo["BIRTHDATE"],
    errors="coerce",
    utc=True
)

# Merge demographics into the ED analytical cohort
ed_cohort = ed_cohort.merge(
    patient_demo,
    left_on="PATIENT",
    right_on="Id",
    how="left"
)

# Ensure index date is timezone-aware
ed_cohort["index_date"] = pd.to_datetime(
    ed_cohort["index_date"],
    errors="coerce",
    utc=True
)

# Calculate age at the index ED encounter
ed_cohort["age_at_index"] = (
    (ed_cohort["index_date"] - ed_cohort["BIRTHDATE"])
    .dt.days / 365.25
).round(1)

print("PATIENT DEMOGRAPHICS")
print("-" * 45)
print(f"Missing birth dates: {ed_cohort['BIRTHDATE'].isna().sum():,}")
print(f"Missing gender: {ed_cohort['GENDER'].isna().sum():,}")
print()
print("Age at index ED encounter:")
print(ed_cohort["age_at_index"].describe().round(1))

ed_cohort[
    [
        "PATIENT",
        "index_date",
        "BIRTHDATE",
        "age_at_index",
        "GENDER",
        "ed_revisit_30d"
    ]
].head(10)

PATIENT DEMOGRAPHICS
---------------------------------------------
Missing birth dates: 0
Missing gender: 0

Age at index ED encounter:
count    263.0
mean      39.9
std       19.7
min        0.0
25%       23.5
50%       40.8
75%       52.9
max       92.6
Name: age_at_index, dtype: float64


,PATIENT,index_date,BIRTHDATE,age_at_index,GENDER,ed_revisit_30d
0,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2018-01-30 01:16:49+00:00,1981-01-20 00:00:00+00:00,37.0,F,False
1,029c95e2-d252-d8d3-3626-7f347182d6d7,2024-06-17 16:49:02+00:00,1972-11-01 00:00:00+00:00,51.6,F,False
2,029c95e2-d252-d8d3-3626-7f347182d6d7,2025-10-05 16:49:02+00:00,1972-11-01 00:00:00+00:00,52.9,F,False
3,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-04-12 15:44:09+00:00,1981-01-08 00:00:00+00:00,14.3,M,False
4,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-06-25 15:44:09+00:00,1981-01-08 00:00:00+00:00,14.5,M,False
5,031ce8e2-1443-01b4-de6b-61a4084a7166,2000-10-16 00:58:03+00:00,1981-01-08 00:00:00+00:00,19.8,M,False
6,031ce8e2-1443-01b4-de6b-61a4084a7166,2009-08-25 23:12:41+00:00,1981-01-08 00:00:00+00:00,28.6,M,False
7,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-07 09:55:05+00:00,1981-01-08 00:00:00+00:00,32.5,M,True
8,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-28 16:55:05+00:00,1981-01-08 00:00:00+00:00,32.6,M,False
9,031ce8e2-1443-01b4-de6b-61a4084a7166,2018-09-23 21:47:05+00:00,1981-01-08 00:00:00+00:00,37.7,M,False


## 11. Prior Healthcare Utilization

Previous healthcare utilization may provide important context for short-term ED reutilization.

For each index ED encounter, utilization features are calculated using only encounters that occurred before the index date. This prevents future information from leaking into the analytical features.

The following measures are constructed:

- Prior ED encounters within 30 days
- Prior ED encounters within 365 days
- Prior healthcare encounters of any type within 365 days

In [25]:
# Ensure encounter timestamps are UTC datetime
encounter_timeline["START"] = pd.to_datetime(
    encounter_timeline["START"],
    errors="coerce",
    utc=True
)

def calculate_prior_utilization(row):
    patient_id = row["PATIENT"]
    index_date = row["index_date"]

    # Patient encounters strictly before the index ED visit
    prior = encounter_timeline[
        (encounter_timeline["PATIENT"] == patient_id)
        & (encounter_timeline["START"] < index_date)
    ]

    # Time windows
    prior_30d = prior[
        prior["START"] >= index_date - pd.Timedelta(days=30)
    ]

    prior_365d = prior[
        prior["START"] >= index_date - pd.Timedelta(days=365)
    ]

    return pd.Series({
        "prior_ed_30d": (
            prior_30d["ENCOUNTERCLASS"] == "emergency"
        ).sum(),

        "prior_ed_365d": (
            prior_365d["ENCOUNTERCLASS"] == "emergency"
        ).sum(),

        "prior_encounters_365d": len(prior_365d)
    })


prior_utilization = ed_cohort.apply(
    calculate_prior_utilization,
    axis=1
)

ed_cohort = pd.concat(
    [
        ed_cohort.reset_index(drop=True),
        prior_utilization.reset_index(drop=True)
    ],
    axis=1
)

print("PRIOR HEALTHCARE UTILIZATION")
print("-" * 50)

print(
    ed_cohort[
        [
            "prior_ed_30d",
            "prior_ed_365d",
            "prior_encounters_365d"
        ]
    ].describe().round(1)
)

ed_cohort[
    [
        "PATIENT",
        "index_date",
        "age_at_index",
        "prior_ed_30d",
        "prior_ed_365d",
        "prior_encounters_365d",
        "ed_revisit_30d"
    ]
].head(15)

PRIOR HEALTHCARE UTILIZATION
--------------------------------------------------
       prior_ed_30d  prior_ed_365d  prior_encounters_365d
count         263.0          263.0                  263.0
mean            0.8            6.2                   13.8
std             1.5           13.2                   17.3
min             0.0            0.0                    0.0
25%             0.0            0.0                    2.0
50%             0.0            0.0                    7.0
75%             1.0            2.0                   15.0
max             4.0           47.0                   74.0


,PATIENT,index_date,age_at_index,prior_ed_30d,prior_ed_365d,prior_encounters_365d,ed_revisit_30d
0,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2018-01-30 01:16:49+00:00,37.0,0,0,14,False
1,029c95e2-d252-d8d3-3626-7f347182d6d7,2024-06-17 16:49:02+00:00,51.6,0,0,2,False
2,029c95e2-d252-d8d3-3626-7f347182d6d7,2025-10-05 16:49:02+00:00,52.9,0,0,2,False
3,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-04-12 15:44:09+00:00,14.3,0,0,0,False
4,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-06-25 15:44:09+00:00,14.5,0,1,1,False
5,031ce8e2-1443-01b4-de6b-61a4084a7166,2000-10-16 00:58:03+00:00,19.8,0,0,0,False
6,031ce8e2-1443-01b4-de6b-61a4084a7166,2009-08-25 23:12:41+00:00,28.6,0,0,2,False
7,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-07 09:55:05+00:00,32.5,0,0,7,True
8,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-28 16:55:05+00:00,32.6,1,1,7,False
9,031ce8e2-1443-01b4-de6b-61a4084a7166,2018-09-23 21:47:05+00:00,37.7,0,0,6,False


## 12. Prior Utilization and 30-Day ED Revisit

Prior healthcare utilization is compared between index ED encounters with and without a subsequent ED revisit within 30 days.

This descriptive comparison evaluates whether recent patterns of healthcare use may help identify patients at higher risk of short-term ED reutilization.

In [26]:
# Compare prior utilization by 30-day ED revisit outcome

utilization_comparison = (
    ed_cohort
    .groupby("ed_revisit_30d")[
        [
            "age_at_index",
            "prior_ed_30d",
            "prior_ed_365d",
            "prior_encounters_365d"
        ]
    ]
    .agg(["count", "mean", "median"])
    .round(1)
)

print("PRIOR UTILIZATION BY 30-DAY ED REVISIT")
print("-" * 55)

utilization_comparison

PRIOR UTILIZATION BY 30-DAY ED REVISIT
-------------------------------------------------------


age_at_index              prior_ed_30d              \
                      count  mean median        count mean median   
ed_revisit_30d                                                      
False                   187  33.8   30.8          187  0.0    0.0   
True                     76  55.1   52.8           76  2.8    3.0   

               prior_ed_365d              prior_encounters_365d               
                       count  mean median                 count  mean median  
ed_revisit_30d                                                                
False                    187   0.2    0.0                   187   5.9    4.0  
True                      76  20.9   18.5                    76  33.2   31.5

In [27]:
# More readable summary

summary = (
    ed_cohort
    .groupby("ed_revisit_30d")
    .agg(
        encounters=("index_encounter_id", "count"),
        mean_age=("age_at_index", "mean"),
        median_prior_ed_30d=("prior_ed_30d", "median"),
        mean_prior_ed_30d=("prior_ed_30d", "mean"),
        median_prior_ed_365d=("prior_ed_365d", "median"),
        mean_prior_ed_365d=("prior_ed_365d", "mean"),
        median_prior_encounters_365d=("prior_encounters_365d", "median"),
        mean_prior_encounters_365d=("prior_encounters_365d", "mean")
    )
    .round(1)
)

summary

,encounters,mean_age,median_prior_ed_30d,mean_prior_ed_30d,median_prior_ed_365d,mean_prior_ed_365d,median_prior_encounters_365d,mean_prior_encounters_365d
ed_revisit_30d,,,,,,,,
False,187,33.8,0.0,0.0,0.0,0.2,4.0,5.9
True,76,55.1,3.0,2.8,18.5,20.9,31.5,33.2


In [28]:
# Quality check: prior ED utilization vs future revisit outcome

check = pd.crosstab(
    ed_cohort["prior_ed_30d"],
    ed_cohort["ed_revisit_30d"],
    margins=True
)

print("PRIOR ED VISITS (30D) VS FUTURE 30-DAY REVISIT")
print("-" * 55)
print(check)

print("\nPatients with NO prior ED visit in 30 days:")
print(
    ed_cohort.loc[
        ed_cohort["prior_ed_30d"] == 0,
        "ed_revisit_30d"
    ].value_counts()
)

print("\nPatients with >=1 prior ED visit in 30 days:")
print(
    ed_cohort.loc[
        ed_cohort["prior_ed_30d"] >= 1,
        "ed_revisit_30d"
    ].value_counts()
)

PRIOR ED VISITS (30D) VS FUTURE 30-DAY REVISIT
-------------------------------------------------------
ed_revisit_30d  False  True  All
prior_ed_30d                    
0                 181     8  189
1                   5     8   13
2                   1     7    8
3                   0    19   19
4                   0    34   34
All               187    76  263

Patients with NO prior ED visit in 30 days:
ed_revisit_30d
False    181
True       8
Name: count, dtype: int64

Patients with >=1 prior ED visit in 30 days:
ed_revisit_30d
True     68
False     6
Name: count, dtype: int64


## 13. Clinical History Before the Index ED Encounter

Clinical conditions documented before each index ED encounter are used to characterize the patient's prior disease burden.

Only conditions with an onset date on or before the index encounter are considered, preserving the temporal structure of the analysis and preventing future clinical information from entering the predictors.

In [29]:
print("CONDITIONS TABLE")
print("-" * 45)

print("Shape:", conditions.shape)

print("\nColumns:")
print(conditions.columns.tolist())

print("\nSample:")
display(conditions.head(10))

CONDITIONS TABLE
---------------------------------------------
Shape: (3517, 7)

Columns:
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE', 'DESCRIPTION']

Sample:


,START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION
0,1996-03-11,NaN,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-13f3-57f4a6b934f9,http://snomed.info/sct,224299000,Received higher education (finding)
1,2006-03-27,2018-01-22,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-1dc5-a353457420c6,http://snomed.info/sct,160903007,Full-time employment (finding)
2,2009-03-30,NaN,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-94b5-26ac8c6c7134,http://snomed.info/sct,714628002,Prediabetes (finding)
3,2009-03-30,NaN,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-94b5-26ac8c6c7134,http://snomed.info/sct,271737000,Anemia (disorder)
4,2009-03-30,NaN,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-94b5-26ac8c6c7134,http://snomed.info/sct,162864005,Body mass index 30+ - obesity (finding)
5,2012-04-02,2018-01-22,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-2a3d-1a1ba491a8ac,http://snomed.info/sct,314529007,Medication review due (situation)
6,2012-04-02,2022-01-31,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-2a3d-1a1ba491a8ac,http://snomed.info/sct,73595000,Stress (finding)
7,2015-04-06,2018-01-22,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-99df-e3a0393fa95e,http://snomed.info/sct,422650009,Social isolation (finding)
8,2016-10-25,2016-11-10,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-25f1-0d3ead2180b6,http://snomed.info/sct,444814009,Viral sinusitis (disorder)
9,2018-01-22,2020-01-27,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-da83-de2c625ae978,http://snomed.info/sct,160904001,Part-time employment (finding)


In [30]:
# Prepare condition dates
conditions_work = conditions.copy()

conditions_work["START"] = pd.to_datetime(
    conditions_work["START"],
    errors="coerce",
    utc=True
)

# Link conditions to each index ED encounter
prior_conditions = ed_cohort[
    ["index_encounter_id", "PATIENT", "index_date"]
].merge(
    conditions_work[
        ["PATIENT", "START", "CODE", "DESCRIPTION"]
    ],
    on="PATIENT",
    how="left"
)

# Keep only conditions documented on or before the index ED encounter
prior_conditions = prior_conditions[
    prior_conditions["START"] <= prior_conditions["index_date"]
].copy()

# Count unique documented conditions before each index encounter
condition_counts = (
    prior_conditions
    .groupby("index_encounter_id")["CODE"]
    .nunique()
    .rename("prior_condition_count")
    .reset_index()
)

# Add feature to analytical cohort
ed_cohort = ed_cohort.merge(
    condition_counts,
    on="index_encounter_id",
    how="left"
)

ed_cohort["prior_condition_count"] = (
    ed_cohort["prior_condition_count"]
    .fillna(0)
    .astype(int)
)

print("PRIOR DOCUMENTED CLINICAL CONDITIONS")
print("-" * 45)

print(ed_cohort["prior_condition_count"].describe().round(1))

ed_cohort[
    [
        "PATIENT",
        "index_date",
        "age_at_index",
        "prior_condition_count",
        "prior_ed_30d",
        "ed_revisit_30d"
    ]
].head(15)

PRIOR DOCUMENTED CLINICAL CONDITIONS
---------------------------------------------
count    263.0
mean      21.2
std       13.2
min        0.0
25%       10.0
50%       18.0
75%       36.0
max       53.0
Name: prior_condition_count, dtype: float64


,PATIENT,index_date,age_at_index,prior_condition_count,prior_ed_30d,ed_revisit_30d
0,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,2018-01-30 01:16:49+00:00,37.0,10,0,False
1,029c95e2-d252-d8d3-3626-7f347182d6d7,2024-06-17 16:49:02+00:00,51.6,22,0,False
2,029c95e2-d252-d8d3-3626-7f347182d6d7,2025-10-05 16:49:02+00:00,52.9,22,0,False
3,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-04-12 15:44:09+00:00,14.3,1,0,False
4,031ce8e2-1443-01b4-de6b-61a4084a7166,1995-06-25 15:44:09+00:00,14.5,2,0,False
5,031ce8e2-1443-01b4-de6b-61a4084a7166,2000-10-16 00:58:03+00:00,19.8,4,0,False
6,031ce8e2-1443-01b4-de6b-61a4084a7166,2009-08-25 23:12:41+00:00,28.6,9,0,False
7,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-07 09:55:05+00:00,32.5,13,0,True
8,031ce8e2-1443-01b4-de6b-61a4084a7166,2013-07-28 16:55:05+00:00,32.6,13,1,False
9,031ce8e2-1443-01b4-de6b-61a4084a7166,2018-09-23 21:47:05+00:00,37.7,18,0,False


In [31]:
# Compare prior documented condition burden by 30-day ED revisit outcome

condition_comparison = (
    ed_cohort
    .groupby("ed_revisit_30d")["prior_condition_count"]
    .agg(["count", "mean", "median", "std"])
    .round(1)
)

print("PRIOR CONDITION BURDEN BY 30-DAY ED REVISIT")
print("-" * 55)
condition_comparison

PRIOR CONDITION BURDEN BY 30-DAY ED REVISIT
-------------------------------------------------------


,count,mean,median,std
ed_revisit_30d,,,,
False,187,14.8,14.0,8.9
True,76,36.8,37.0,7.9


In [32]:
# Most common prior documented conditions by 30-day ED revisit outcome

condition_outcomes = prior_conditions.merge(
    ed_cohort[
        ["index_encounter_id", "ed_revisit_30d"]
    ],
    on="index_encounter_id",
    how="left"
)

top_conditions = (
    condition_outcomes
    .groupby(["DESCRIPTION", "ed_revisit_30d"])["index_encounter_id"]
    .nunique()
    .unstack(fill_value=0)
)

top_conditions.columns = ["no_revisit", "revisit"]

top_conditions["total"] = (
    top_conditions["no_revisit"] +
    top_conditions["revisit"]
)

top_conditions = (
    top_conditions
    .sort_values("total", ascending=False)
    .head(25)
)

print("MOST COMMON PRIOR DOCUMENTED CONDITIONS")
print("-" * 50)

top_conditions

MOST COMMON PRIOR DOCUMENTED CONDITIONS
--------------------------------------------------


,no_revisit,revisit,total
DESCRIPTION,,,
Medication review due (situation),161,76,237
Full-time employment (finding),129,75,204
Stress (finding),118,74,192
Gingivitis (disorder),91,73,164
Part-time employment (finding),81,74,155
Body mass index 30+ - obesity (finding),76,66,142
Social isolation (finding),76,66,142
Anemia (disorder),69,72,141
Victim of intimate partner abuse (finding),55,72,127


In [33]:
# Quality check: clinical conditions strongly associated with revisit group

conditions_to_check = [
    "Essential hypertension (disorder)",
    "Metabolic syndrome X (disorder)",
    "Chronic kidney disease stage 1 (disorder)",
    "Disorder of kidney due to diabetes mellitus (disorder)",
    "Ischemic heart disease (disorder)",
    "Anemia (disorder)"
]

clinical_check = []

for condition in conditions_to_check:
    
    patients_with_condition = prior_conditions.loc[
        prior_conditions["DESCRIPTION"] == condition,
        "index_encounter_id"
    ].unique()
    
    temp = ed_cohort[
        ed_cohort["index_encounter_id"].isin(patients_with_condition)
    ]
    
    clinical_check.append({
        "condition": condition,
        "index_encounters": len(temp),
        "unique_patients": temp["PATIENT"].nunique(),
        "revisits_30d": temp["ed_revisit_30d"].sum(),
        "revisit_rate_pct": temp["ed_revisit_30d"].mean() * 100
    })

clinical_check = pd.DataFrame(clinical_check)

clinical_check["revisit_rate_pct"] = (
    clinical_check["revisit_rate_pct"].round(1)
)

clinical_check

,condition,index_encounters,unique_patients,revisits_30d,revisit_rate_pct
0,Essential hypertension (disorder),98,16,67,68.4
1,Metabolic syndrome X (disorder),94,8,72,76.6
2,Chronic kidney disease stage 1 (disorder),89,6,73,82.0
3,Disorder of kidney due to diabetes mellitus (disorder),89,6,73,82.0
4,Ischemic heart disease (disorder),86,7,70,81.4
5,Anemia (disorder),141,29,72,51.1


In [34]:
# Patient-level concentration of ED utilization and 30-day revisits

patient_utilization = (
    ed_cohort
    .groupby("PATIENT")
    .agg(
        index_ed_encounters=("index_encounter_id", "count"),
        revisits_30d=("ed_revisit_30d", "sum"),
        mean_age=("age_at_index", "mean"),
        max_prior_conditions=("prior_condition_count", "max"),
        max_prior_ed_365d=("prior_ed_365d", "max")
    )
    .reset_index()
)

patient_utilization = patient_utilization.sort_values(
    "index_ed_encounters",
    ascending=False
)

print("PATIENT-LEVEL ED UTILIZATION CONCENTRATION")
print("-" * 55)

print(f"Unique patients: {len(patient_utilization):,}")
print(
    f"Patients with >1 index ED encounter: "
    f"{(patient_utilization['index_ed_encounters'] > 1).sum():,}"
)
print(
    f"Patients with >=1 30-day revisit: "
    f"{(patient_utilization['revisits_30d'] > 0).sum():,}"
)

patient_utilization.head(15)

PATIENT-LEVEL ED UTILIZATION CONCENTRATION
-------------------------------------------------------
Unique patients: 82
Patients with >1 index ED encounter: 52
Patients with >=1 30-day revisit: 8


,PATIENT,index_ed_encounters,revisits_30d,mean_age,max_prior_conditions,max_prior_ed_365d
35,5693c080-f485-91e7-54b9-ad9a8c12af62,61,57,52.021311,37,47
62,bca1691f-8839-1d66-ed01-471134d55738,11,8,79.190909,53,7
22,355a4d50-2628-aa65-c336-deacf4d606fb,10,5,64.200000,40,5
12,2211f478-b7b4-7711-16cf-84ffb52b9d2b,9,0,23.277778,40,1
20,31f368d9-3fdc-a15f-ccdc-fafa74a860e9,8,0,28.500000,27,2
2,031ce8e2-1443-01b4-de6b-61a4084a7166,7,1,25.714286,18,1
53,96de62ae-1b00-58d8-77f9-cd25414e16be,6,0,22.833333,21,1
55,9c5b16dc-8a47-896c-4d74-f96f8c98891b,5,0,41.940000,20,1
21,3314e296-2326-2826-271d-a7be5b896db9,5,0,30.820000,18,1
15,2e9a9dab-f5b6-8960-d3ce-7834bb5e7344,4,0,30.050000,18,1


In [35]:
# Quantify concentration of 30-day ED revisits across patients

total_revisits = patient_utilization["revisits_30d"].sum()

revisit_concentration = (
    patient_utilization[
        patient_utilization["revisits_30d"] > 0
    ][["PATIENT", "index_ed_encounters", "revisits_30d"]]
    .sort_values("revisits_30d", ascending=False)
    .copy()
)

revisit_concentration["share_of_all_revisits_pct"] = (
    revisit_concentration["revisits_30d"]
    / total_revisits
    * 100
).round(1)

revisit_concentration["cumulative_share_pct"] = (
    revisit_concentration["revisits_30d"]
    .cumsum()
    / total_revisits
    * 100
).round(1)

print("30-DAY ED REVISIT CONCENTRATION")
print("-" * 50)
print(f"Total 30-day revisit events: {total_revisits:,}")
print(f"Patients generating revisits: {len(revisit_concentration):,}")
print()

revisit_concentration

30-DAY ED REVISIT CONCENTRATION
--------------------------------------------------
Total 30-day revisit events: 76
Patients generating revisits: 8



,PATIENT,index_ed_encounters,revisits_30d,share_of_all_revisits_pct,cumulative_share_pct
35,5693c080-f485-91e7-54b9-ad9a8c12af62,61,57,75.0,75.0
62,bca1691f-8839-1d66-ed01-471134d55738,11,8,10.5,85.5
22,355a4d50-2628-aa65-c336-deacf4d606fb,10,5,6.6,92.1
36,57efda89-b582-bb08-8a2d-e06b2c184bfc,3,2,2.6,94.7
2,031ce8e2-1443-01b4-de6b-61a4084a7166,7,1,1.3,96.1
49,8b9df7af-24b3-66ee-afeb-2734ebcbd33c,4,1,1.3,97.4
5,0b786670-17af-32e1-b2d2-f77c33874b30,4,1,1.3,98.7
77,f6359fe4-4340-2820-8b76-593a45f5186c,4,1,1.3,100.0


In [36]:
# Inspect encounter sequence for the highest-utilization patient

top_patient = revisit_concentration.iloc[0]["PATIENT"]

top_patient_journey = (
    ed_cohort.loc[
        ed_cohort["PATIENT"] == top_patient,
        [
            "PATIENT",
            "index_date",
            "prior_ed_30d",
            "prior_ed_365d",
            "prior_encounters_365d",
            "ed_revisit_30d"
        ]
    ]
    .sort_values("index_date")
    .reset_index(drop=True)
)

top_patient_journey["days_since_previous_index"] = (
    top_patient_journey["index_date"]
    .diff()
    .dt.total_seconds()
    / 86400
).round(1)

print("HIGHEST-UTILIZATION PATIENT JOURNEY")
print("-" * 50)
print(f"Index ED encounters: {len(top_patient_journey)}")
print()

top_patient_journey

HIGHEST-UTILIZATION PATIENT JOURNEY
--------------------------------------------------
Index ED encounters: 61



,PATIENT,index_date,prior_ed_30d,prior_ed_365d,prior_encounters_365d,ed_revisit_30d,days_since_previous_index
0,5693c080-f485-91e7-54b9-ad9a8c12af62,2003-12-25 23:50:14+00:00,0,0,0,False,NaN
1,5693c080-f485-91e7-54b9-ad9a8c12af62,2017-01-01 23:50:14+00:00,0,0,2,False,4756.0
2,5693c080-f485-91e7-54b9-ad9a8c12af62,2019-10-08 00:14:42+00:00,0,0,6,False,1009.0
3,5693c080-f485-91e7-54b9-ad9a8c12af62,2022-06-12 23:50:14+00:00,0,0,8,False,979.0
4,5693c080-f485-91e7-54b9-ad9a8c12af62,2025-04-13 23:50:14+00:00,0,0,4,True,1036.0
...,...,...,...,...,...,...,...
56,5693c080-f485-91e7-54b9-ad9a8c12af62,2026-05-31 23:50:14+00:00,3,47,52,True,7.0
57,5693c080-f485-91e7-54b9-ad9a8c12af62,2026-06-07 23:50:14+00:00,3,47,52,True,7.0
58,5693c080-f485-91e7-54b9-ad9a8c12af62,2026-06-21 23:50:14+00:00,3,47,52,True,14.0
59,5693c080-f485-91e7-54b9-ad9a8c12af62,2026-06-28 23:50:14+00:00,3,47,52,True,7.0


In [37]:
# Inspect recent high-utilization period for the top patient

recent_top_patient = top_patient_journey[
    top_patient_journey["index_date"] >= "2025-01-01"
].copy()

print("RECENT HIGH-UTILIZATION PERIOD")
print("-" * 50)
print(f"ED encounters since 2025: {len(recent_top_patient):,}")
print()

recent_top_patient[
    [
        "index_date",
        "days_since_previous_index",
        "prior_ed_30d",
        "ed_revisit_30d"
    ]
]

RECENT HIGH-UTILIZATION PERIOD
--------------------------------------------------
ED encounters since 2025: 57



,index_date,days_since_previous_index,prior_ed_30d,ed_revisit_30d
4,2025-04-13 23:50:14+00:00,1036.0,0,True
5,2025-04-27 23:50:14+00:00,14.0,1,True
6,2025-05-04 23:50:14+00:00,7.0,2,True
7,2025-05-11 23:50:14+00:00,7.0,3,True
8,2025-05-25 23:50:14+00:00,14.0,3,True
9,2025-06-01 23:50:14+00:00,7.0,3,True
10,2025-06-15 23:50:14+00:00,14.0,2,True
11,2025-06-22 23:50:14+00:00,7.0,3,True
12,2025-06-29 23:50:14+00:00,7.0,3,True
13,2025-07-06 23:50:14+00:00,7.0,3,True


In [38]:
# Compare revisit patterns across patients with 30-day revisits

revisit_patient_check = (
    ed_cohort[ed_cohort["ed_revisit_30d"]]
    .groupby("PATIENT")
    .agg(
        revisit_events=("ed_revisit_30d", "sum"),
        first_revisit_index=("index_date", "min"),
        last_revisit_index=("index_date", "max"),
        max_prior_ed_30d=("prior_ed_30d", "max"),
        max_prior_ed_365d=("prior_ed_365d", "max")
    )
    .sort_values("revisit_events", ascending=False)
)

revisit_patient_check["share_of_revisits_pct"] = (
    revisit_patient_check["revisit_events"]
    / revisit_patient_check["revisit_events"].sum()
    * 100
).round(1)

print("PATIENT-LEVEL REVISIT PATTERN CHECK")
print("-" * 50)

revisit_patient_check

PATIENT-LEVEL REVISIT PATTERN CHECK
--------------------------------------------------


,revisit_events,first_revisit_index,last_revisit_index,max_prior_ed_30d,max_prior_ed_365d,share_of_revisits_pct
PATIENT,,,,,,
5693c080-f485-91e7-54b9-ad9a8c12af62,57,2025-04-13 23:50:14+00:00,2026-07-05 23:50:14+00:00,4,47,75.0
bca1691f-8839-1d66-ed01-471134d55738,8,2026-04-15 11:39:55+00:00,2026-07-15 11:39:55+00:00,4,7,10.5
355a4d50-2628-aa65-c336-deacf4d606fb,5,2022-02-03 20:04:51+00:00,2022-03-31 20:04:51+00:00,3,4,6.6
57efda89-b582-bb08-8a2d-e06b2c184bfc,2,2017-09-27 06:49:34+00:00,2017-10-04 06:49:34+00:00,1,1,2.6
031ce8e2-1443-01b4-de6b-61a4084a7166,1,2013-07-07 09:55:05+00:00,2013-07-07 09:55:05+00:00,0,0,1.3
0b786670-17af-32e1-b2d2-f77c33874b30,1,2014-09-30 23:29:08+00:00,2014-09-30 23:29:08+00:00,0,0,1.3
8b9df7af-24b3-66ee-afeb-2734ebcbd33c,1,2023-01-05 03:54:37+00:00,2023-01-05 03:54:37+00:00,0,0,1.3
f6359fe4-4340-2820-8b76-593a45f5186c,1,2024-12-30 00:54:50+00:00,2024-12-30 00:54:50+00:00,0,0,1.3


### Key Finding: Revisit Concentration and Repeated Measures

Thirty-day ED revisits were highly concentrated among a small number of patients. Only 8 unique patients generated the 76 revisit events observed in the encounter-level cohort, and a single high-utilization patient accounted for 57 events (75% of all revisits). The three highest-utilization patients together accounted for approximately 92% of revisit events.

This concentration indicates that encounter-level comparisons are strongly influenced by repeated observations from a small number of patients. Therefore, associations between prior utilization, clinical burden, and subsequent ED revisits should not be interpreted as independent patient-level risk relationships.

The longitudinal encounter analysis is retained to characterize utilization patterns and demonstrate the structure of the EHR data. Subsequent analyses will distinguish patient-level characteristics from encounter-level utilization and will explicitly account for the concentration of repeated encounters among high-utilization patients.

Because the dataset is synthetic, highly regular utilization patterns—such as repeated encounters occurring at short, consistent intervals—may reflect characteristics of the data-generation process rather than real-world clinical behavior.